In [1]:
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')

import numpy as np
import pandas as pd
import pyterrier as pt
import os
import ir_datasets
from urllib.parse import urlparse, parse_qs
import re
from sklearn.model_selection import train_test_split
import copy

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from keras.layers import Input
import time
from sklearn.metrics.pairwise import cosine_similarity

2025-01-18 20:48:50.798019: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-18 20:48:50.808407: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-18 20:48:50.928133: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-18 20:48:51.047811: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1737222531.146707    4629 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1737222531.17

In [2]:
if not pt.started():
    pt.init()

/tmp/ipykernel_4629/3057724015.py:1: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_4629/3057724015.py:2: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [3]:
dataset = ir_datasets.load('istella22/test')

# CREATION OF WEBSITE GAS MODEL

In [ ]:

# takes toooooooo much time
if False:
    num_of_docs = len(dataset.docs)
    
    
    docno_to_index = {}
    erronious_indexes = []
    for index in range(num_of_docs):
        try:
            docno_to_index[dataset.docs[index].doc_id] = index
        except:
            erronious_indexes.append(index)
            print("error at:", index)
            continue

In [17]:
INDEX_EXISTS = True
index_dir = '/media/ersel/Expansion/istella22_index3'

if INDEX_EXISTS:
    index = pt.IndexFactory.of(index_dir)
else:
    def doc_to_dict_generator(docs):
        global error_no
        for doc in docs:
            try:
                yield {"docno": doc.doc_id, "text": doc.text}
            except:
                yield {"docno": str(error_no), "text": "istella document error"}
                error_no -= 1
    
    indexer = pt.index.IterDictIndexer(index_dir)
    index = indexer.index(doc_to_dict_generator(dataset.docs))


In [18]:
bm25_retriever = pt.BatchRetrieve(index, wmodel="BM25") % 100

/tmp/ipykernel_8401/3716249422.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25_retriever = pt.BatchRetrieve(index, wmodel="BM25") % 100


In [19]:
search_res = bm25_retriever.search("sad life")
print(search_res)
print(len(search_res))

   qid    docid             docno  rank      score     query
0    1  2775598  1990011300499881     0  23.154382  sad life
1    1  1935575  1990010900176593     1  23.025193  sad life
2    1  3032524  1990011401260999     2  22.925853  sad life
3    1   406548  1990010102281031     3  22.835376  sad life
4    1  1021330  1990010401842904     4  22.354891  sad life
..  ..      ...               ...   ...        ...       ...
95   1   857857  1990010302557020    95  19.702936  sad life
96   1  1222075  1990010501697919    96  19.694342  sad life
97   1   716068  1990010300627743    97  19.647187  sad life
98   1  2889559  1990011302137931    98  19.636935  sad life
99   1  3323228  1990011502613031    99  19.631194  sad life

[100 rows x 6 columns]
100


Istella22Doc(doc_id='1990011300499881', title='A life story | Sad Video | Lover of Sadness', url='http://www.loverofsadness.net/sad_video.php?id=74&o=1', text="Lover of Sadness Contact | Suggestion | Claim Credit | Terms & Condition | Privacy Policy Copyright © all rights reserverd 2009 A Life Story Description: A life story of a guy and his loving family Sad Sad with esothori plzz can to laefi Sadness Life video &lt;&lt; Previous Video Music Video Next Video >> Nepeta says: 16 Nov, 2014 11:52 PM I cried at the dog part. Noooooooo!!! Dogg!!!!!! ;,( satyaranjan sahoo says: 30 Jan, 2015 01:29 PM it really heart touchinng..... Do not post other site's link, it will be considered as spam Umm, are you really just giving this info out for nointhg? Created by Bony Yousuf - From Gloomy Sunday My Account | Logout I miss u shona... Register | Login Tags: Life , Love , Death , Animal 2 Post a Comment Inspirational A life story Awesome!!! Separation Lost Love Sacrifice Wedding Suicide Missing Pare

In [12]:
queries = {}
relevant_set = set()
for query in dataset.queries:
    queries[query.query_id] = {"text": query.text, "docs":{}}
for qrel in dataset.qrels:
    queries[qrel.query_id]['docs'][qrel.doc_id] = qrel.relevance
    relevant_set.add(qrel.doc_id)


In [13]:
def dcg_at_k(relevances, k=None):
    if k:
        relevances = relevances[:k]
    return sum([rel / np.log2(i + 1) for i, rel in enumerate(relevances, 1)])

def ndcg_at_k(relevances, k=None):
    dcg = dcg_at_k(relevances, k)
    idcg = dcg_at_k(sorted(relevances, reverse=True), k)
    return dcg / idcg if idcg > 0 else 0

In [14]:
queries2 = {} 
for qid, q_dict in queries.items():
    # qid, q_dict = '263', queries['263']
    try:
        bm25_res = bm25_retriever.search(q_dict['text'])
    except:
        continue
    doc_dict = q_dict['docs']
    state_flag = False
    for doc in bm25_res.iloc:
        if doc.docno in doc_dict:
            state_flag = True
            break
    if state_flag:
        q_dict['bm25'] = bm25_res
        queries2[qid] = q_dict

# FEATURE FUNCTIONS

In [4]:
def title_query_term_overlap(doc, query):
    title_terms = set(doc.title.split())
    query_terms = set(query.split())
    return len(title_terms & query_terms)

def title_query_jaccard_similarity(doc, query):
    title_terms = set(doc.title.split())
    query_terms = set(query.split())
    intersection = len(title_terms & query_terms)
    union = len(title_terms | query_terms)
    return intersection / union if union else 0

def title_query_dice_similarity(doc, query):
    title_terms = set(doc.title.split())
    query_terms = set(query.split())
    intersection = len(title_terms & query_terms)
    return (2 * intersection) / (len(title_terms) + len(query_terms)) if title_terms and query_terms else 0

def title_query_position(doc, query):
    title_words = doc.title.split()
    query_terms = query.split()
    for i, word in enumerate(title_words):
        if word in query_terms:
            return i
    return -1

def exact_match_title_query(doc, query):
    return 1 if doc.title.strip().lower() == query.strip().lower() else 0

def query_length(query):
    return len(query.split())

def query_character_length(query):
    return len(query)

def term_overlap(doc, query):
    query_terms = set(query.split())
    doc_terms = set(doc.text.split())
    return len(query_terms & doc_terms)

def jaccard_similarity(doc, query):
    query_terms = set(query.split())
    doc_terms = set(doc.text.split())
    intersection = len(query_terms & doc_terms)
    union = len(query_terms | doc_terms)
    return intersection / union if union else 0

def dice_similarity(doc, query):
    query_terms = set(query.split())
    doc_terms = set(doc.text.split())
    intersection = len(query_terms & doc_terms)
    return (2 * intersection) / (len(query_terms) + len(doc_terms)) if query_terms and doc_terms else 0

def term_overlap_extra(doc, query):
    query_terms = set(query.split())
    doc_terms = set(doc.extra_text.split())
    return len(query_terms & doc_terms)

def jaccard_similarity_extra(doc, query):
    query_terms = set(query.split())
    doc_terms = set(doc.extra_text.split())
    intersection = len(query_terms & doc_terms)
    union = len(query_terms | doc_terms)
    return intersection / union if union else 0

def dice_similarity_extra(doc, query):
    query_terms = set(query.split())
    doc_terms = set(doc.extra_text.split())
    intersection = len(query_terms & doc_terms)
    return (2 * intersection) / (len(query_terms) + len(doc_terms)) if query_terms and doc_terms else 0









def document_length(doc):
    return len(doc.text.split())

def document_character_length(doc):
    return len(doc.text)

def average_sentence_length(doc):
    sentences = re.split(r'[.!?]', doc.text)
    sentences = [sent.strip() for sent in sentences if sent.strip()]  # Remove empty sentences and leading/trailing spaces
    return sum(len(sent.split()) for sent in sentences) / len(sentences) if sentences else 0

def stopword_proportion(doc):
    words = doc.split()
    stopword_count = sum(1 for word in words if word.lower() in STOPWORDS)
    return stopword_count / len(words) if words else 0

def unique_word_count(doc):
    return len(set(doc.text.split()))


def url_depth(doc):
    return doc.url.count('/')

def has_query_parameters(doc):
    return '?' in doc.url

def get_features(doc, query):
    return [
        title_query_term_overlap(doc, query),
        title_query_jaccard_similarity(doc, query),
        title_query_dice_similarity(doc, query),
        title_query_position(doc, query),
        exact_match_title_query(doc, query),
        query_length(query),
        query_character_length(query),
        term_overlap(doc, query),
        jaccard_similarity(doc, query),
        dice_similarity(doc, query),
        term_overlap_extra(doc, query),
        jaccard_similarity_extra(doc, query),
        dice_similarity_extra(doc, query),
        document_length(doc),
        document_character_length(doc),
        average_sentence_length(doc),
        unique_word_count(doc),
        url_depth(doc),
        has_query_parameters(doc),
    ]

In [ ]:
for qid, q_dict in queries2.items():
    q_text = q_dict['text']
    q_dict['features'] = []
    for row in q_dict['bm25'].iloc:
        bm25_score = row['score']
        docindex = int(row['docid'])
        doc = dataset.docs[docindex]
        query = q_dict['text']
        qd_features = get_features(doc, query)
        qd_features.append(bm25_score)
        q_dict['features'].append(qd_features)

In [ ]:
for qid, q_dict in queries2.items():
    q_dict['scores'] = []
    for row in q_dict['bm25'].iloc:
        score = 0
        if row.docno in q_dict['docs']:
            score = q_dict['docs'][row.docno]
        q_dict['scores'].append(score)

# Shared Funcs

In [5]:
def dcg_at_k(relevances, k=None):
    if k:
        relevances = relevances[:k]
    return sum([rel / np.log2(i + 1) for i, rel in enumerate(relevances, 1)])

def ndcg_at_k(relevances, k=None):
    dcg = dcg_at_k(relevances, k)
    idcg = dcg_at_k(sorted(relevances, reverse=True), k)
    return dcg / idcg if idcg > 0 else 0

In [6]:
def normal_access(container, i1, i2):
    return container[i1][i2]

def csr_matrix_access(container, i1, i2):
    return container[(i1, i2)]

In [7]:
def kendalls_tau(f1_index, f2_index, features_list, access=normal_access):
    concordont_count = 0
    discordont_count = 0
    try:
        n = len(features_list)
    except:
        n = features_list.shape[0]
    for d1 in range(n):
        d1_f1 = access(features_list, d1, f1_index)
        d1_f2 = access(features_list, d1, f2_index)
        for d2 in range(d1+1, n):
            d2_f1 = access(features_list, d2, f1_index)
            d2_f2 = access(features_list, d2, f2_index)
            if (
                    (
                        (d1_f1 > d2_f1)
                        and
                        (d1_f2 > d2_f2)
                    ) or
                    (
                        (d1_f1 < d2_f1)
                        and
                        (d1_f2 < d2_f2)
                    )
            ):
                concordont_count += 1
            elif (
                    (
                        (d1_f1 > d2_f1)
                        and
                        (d1_f2 < d2_f2)
                    ) or
                    (
                        (d1_f1 < d2_f1)
                        and
                        (d1_f2 > d2_f2)
                    )
            ):
                discordont_count += 1
    return (concordont_count - discordont_count) / (n*(n-1)/2)

In [8]:
def get_sim_key(f1, f2):
    return str(min(f1, f2)) + "#" + str(max(f1, f2))

In [9]:
def get_best_feature(features):
    best_feature = None
    best_feature_index = None
    for index, feature in features:
        if not best_feature or feature >= best_feature:
            best_feature = feature
            best_feature_index = index 
    return best_feature_index, best_feature

def update_features(features, best_feature_index, similarity_dict):
    for feature_tuple in features:
        feature_index, feature_score = feature_tuple
        if feature_index != best_feature_index:
            key = get_sim_key(feature_index, best_feature_index)
            sim_val = similarity_dict[key]
            feature_tuple[1] -= sim_val

def GAS(features, similarity_dict):
    features2 = [[index, score] for index, score in enumerate(features)]
    n = len(features2)
    features_ordered = []
    for i in range(n-1):
        best_feature_index, best_feature_score = get_best_feature(features2)
        print(best_feature_index, best_feature_score)
        features_ordered.append(best_feature_index)
        update_features(features2, best_feature_index, similarity_dict)
        features2 = [feature_tuple for feature_tuple in features2 if feature_tuple[0] != best_feature_index]
    features_ordered.append(features2[0][0])
    return features_ordered

# LAMBDAMART

In [10]:
import lightgbm as lgb
from sklearn.datasets import load_svmlight_file

In [11]:
#from rankeval.dataset import Dataset
#from rankeval.model import RTEnsemble
TEST_FILE = '/media/ersel/Expansion/code/CENG778-istella22_trials/lambdamart/data/test.monoT5.svm'
MODEL_FILE = '/media/ersel/Expansion/code/CENG778-istella22_trials/lambdamart/models/lambdamart.monoT5.lgb'

In [12]:
X, y, q = load_svmlight_file(TEST_FILE, query_id=True)

In [13]:

divide_point = 1074050

if divide_point is None:
    old_qid = q[0]
    qid_count = 1
    for index, qid in enumerate(q):
        if qid != old_qid:
            qid_count += 1
            old_qid = qid
            if qid_count == 1501:
                divide_point = index
                break

X_train = X[:divide_point]
X_test = X[divide_point:]
y_train = y[:divide_point]
y_test = y[divide_point:]
q_train = q[:divide_point]
q_test = q[divide_point:]
print(divide_point)

1074050


In [14]:
qid_limits_train = []
old_qid = q_train[0]
start_index = 0
for index, qid in enumerate(q_train):
    if qid != old_qid:
        old_qid = qid
        qid_limits_train.append((start_index, index))
        start_index = index
qid_limits_train.append((start_index, index + 1))

qid_limits_test = []
old_qid = q_test[0]
start_index = 0
for index, qid in enumerate(q_test):
    if qid != old_qid:
        old_qid = qid
        qid_limits_test.append((start_index, index))
        start_index = index
qid_limits_test.append((start_index, index + 1))

In [15]:
print(len(set(q_train)))
print(len(set(q_test)))
print(len(set(qid_limits_train)))
print(len(set(qid_limits_test)))
divide_point

1500
698
1500
698


1074050

In [16]:
lgbm_lmart = lgb.Booster(model_file=MODEL_FILE)
predictions = lgbm_lmart.predict(X_test)

In [17]:
def generate_qid_data():
    old_qid = q_test[0]
    pairs = []
    for qid, pred_val, true_val in zip(q_test, predictions, y_test):
        if qid != old_qid:
            yield old_qid, pairs
            old_qid = qid
            pairs = []
        pairs.append((pred_val, true_val))
    yield qid, pairs

In [18]:
average_ndcg = 0
num_of_qs = 0
for qid, pairs in generate_qid_data():
    num_of_qs += 1
    pairs = sorted(pairs, key=lambda x: x[0], reverse=True)
    pair_scores = [pair[1] for pair in pairs]
    average_ndcg += ndcg_at_k(pair_scores)
average_ndcg = average_ndcg/num_of_qs

In [19]:
print(average_ndcg)

0.8747334768005786


In [21]:
start, end = qid_limits_train[0]
q_train[start: end]

array([263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263,
       263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263,
       263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263,
       263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263,
       263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263,
       263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263,
       263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263,
       263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263,
       263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263,
       263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263,
       263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263,
       263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263,
       263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263,
       263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 263, 26

In [20]:
"""
start, end = qid_limits_train[0]
start_time = time.time()
kendalls_tau(1,2,X_train[start: end],access=csr_matrix_access)
print(time.time() - start_time)
"""

'\nstart, end = qid_limits_train[0]\nstart_time = time.time()\nkendalls_tau(1,2,X_train[start: end],access=csr_matrix_access)\nprint(time.time() - start_time)\n'

In [31]:
list_of_elements = X_test[start:end, f1].toarray().flatten().tolist()

In [33]:
len

999

In [47]:
start, end = (0,999)
f1 = 1
f2 = 2
print([X_test[start:end,f1].toarray().flatten().tolist()])
start_time = time.time()
sim_score = cosine_similarity([X_test[start:end,f1].toarray().flatten().tolist()], [X_test[start:end,f2].toarray().flatten().tolist()])
print(X_test[start:end,f1].shape)
print('time:', time.time() - start_time)
print('sim_score', sim_score)
print(X_test[start:end,f1].shape)
print(X_test[:,5][(0,0)])
print(X_test[:,5][(3,0)])

print(X_test[(0,5)])
print(X_test[(3,5)])

[[10111.0, 10198.0, 10241.0, 10362.0, 1049.0, 10565.0, 10750.0, 18136.0, 18143.0, 18185.0, 18186.0, 18229.0, 18290.0, 18417.0, 18470.0, 18946.0, 19105.0, 1922.0, 20220.0, 219.0, 2764.0, 289.0, 3194.0, 3204.0, 36795.0, 3681.0, 4105.0, 4256.0, 4355.0, 5295.0, 6727.0, 950.0, 9953.0, 9969.0, 23276.0, 23430.0, 24011.0, 25027.0, 2091.0, 1702.0, 1706.0, 10401.0, 911.0, 910.0, 638.0, 781.0, 352.0, 3160.0, 478.0, 11005.0, 11063.0, 11392.0, 11526.0, 11777.0, 11905.0, 11908.0, 11940.0, 12005.0, 12072.0, 12076.0, 12267.0, 12495.0, 12543.0, 12693.0, 12697.0, 12767.0, 13113.0, 13121.0, 13130.0, 13223.0, 13238.0, 13318.0, 13321.0, 13338.0, 13397.0, 13403.0, 13453.0, 13454.0, 13466.0, 13512.0, 13578.0, 1359.0, 13637.0, 13829.0, 13831.0, 13895.0, 13933.0, 13961.0, 14017.0, 14043.0, 14152.0, 14190.0, 14266.0, 14710.0, 14733.0, 14771.0, 14774.0, 14784.0, 14794.0, 14831.0, 14889.0, 15081.0, 15135.0, 15159.0, 15240.0, 15251.0, 15254.0, 15275.0, 15308.0, 15314.0, 15317.0, 15346.0, 15366.0, 15400.0, 15413.0,

In [56]:
feature_count = X_train.shape[1]
train_query_count = 1500
similarity_dict = {
    get_sim_key(f1_index, f2_index): 0
    for f1_index in range(feature_count)
    for f2_index in range(f1_index + 1, feature_count)
}



start_time = 0
for i in range(train_query_count):
    start, end = qid_limits_train[i]
    print(i, 'start,end:', start, end, 'time:', time.time() - start_time)
    start_time = time.time()
    for f1_index in range(feature_count):
        for f2_index in range(f1_index + 1, feature_count):
            key = get_sim_key(f1_index, f2_index)
            f1_container = X_train[start:end, f1_index].toarray().flatten().tolist()
            f2_container = X_train[start:end, f2_index].toarray().flatten().tolist()
            score = cosine_similarity([f1_container], [f2_container])[0][0]
            similarity_dict[key] += score

0 start,end: 0 999 time: 1737223821.9433124
1 start,end: 999 1999 time: 24.482168912887573
2 start,end: 1999 2406 time: 23.557868719100952
3 start,end: 2406 3406 time: 14.839759588241577
4 start,end: 3406 4406 time: 24.487834453582764


KeyboardInterrupt: 

In [43]:
feature_count = X_train.shape[1]
feature_count

221

In [14]:
qid_set = {q for q in q_test}
len(qid_set)

2198

In [77]:
x = {}
c=0
for i in y_test:
    x.setdefault(i, 0)
    x[i] += 1
x

{np.float64(0.0): 1491011,
 np.float64(3.0): 2573,
 np.float64(4.0): 1040,
 np.float64(1.0): 6070,
 np.float64(2.0): 1010}

In [56]:
len(dataset.qrels)

10693

10693

In [36]:
type(X_test)

scipy.sparse._csr.csr_matrix

In [33]:
type(X_test)

scipy.sparse._csr.csr_matrix

In [44]:
kendalls_tau(3, 5, X_test, access=csr_matrix_access)

KeyboardInterrupt: 

In [16]:
for qid,q_dict in queries2.items():
    similarity_dict = q_dict['similarity_dict'] = {}
    feature_count = len(q_dict['features'][0])
    for f1_index in range(feature_count):
        for f2_index in range(f1_index + 1, feature_count):
            key = get_sim_key(f1_index, f2_index)
            similarity_dict[key] = kendalls_tau(f1_index, f2_index, q_dict['features'])

In [17]:
for qid,q_dict in queries2.items():
    n = len(q_dict['features'][0])

    q_dict['feature_scores'] = []
    for f_index in range(feature_count):
        c_feature_list = [
            (
                feature_list[f_index],
                score
            ) for feature_list, score in zip(q_dict['features'], q_dict['scores'])
        ]
        feature_sorted_list = [second for _, second in sorted(c_feature_list, key=lambda x: x[0])]
        ndcg_score = ndcg_at_k(feature_sorted_list)
        feature_sorted_list.reverse()
        ndcg_score2 = ndcg_at_k(feature_sorted_list)
        q_dict['feature_scores'].append(max(ndcg_score,ndcg_score2))


In [19]:
size1 = 50
size2 = 25
q_list = list(queries2.items())

train_queries = dict(q_list[:size1])
test_queries = dict(q_list[size1:size1 + size2])

In [20]:
feature_scores = None
similarity_dict = None

for qid, q_dict in train_queries.items():
    if feature_scores is None:
        feature_scores = copy.deepcopy(q_dict['feature_scores'])
        similarity_dict = copy.deepcopy(q_dict['similarity_dict'])
    else:
        feature_scores = [i + j for i, j in zip(feature_scores, q_dict['feature_scores'])]
        for key, score in similarity_dict.items():
            similarity_dict[key] = score + q_dict['similarity_dict'][key]

feature_scores = [score/size1 for score in feature_scores]
similarity_dict = {key: score/size1 for key, score in similarity_dict.items()}
features_ordered = GAS(feature_scores, similarity_dict)

19 0.5535662359743142
6 0.5535662359743142
5 0.5535662359743142
4 0.5523298723379506
18 0.5462383935264336
17 0.5664910968908279
2 0.5377993904053565
16 0.36168211673876977
9 0.5608158411803958
15 0.27588818955505623
0 0.18285651076666448
12 0.020630412381092367
14 -0.21765881150305547
8 -0.13577405780950308
1 -0.31572788232191606
3 -0.5593067386169553
10 -0.8262751733749241
13 -0.8744934440867447
7 -1.3262980661234671


In [129]:
print(features_ordered)

[19, 6, 5, 4, 18, 17, 2, 16, 9, 15, 0, 12, 14, 8, 1, 3, 10, 13, 7, 11]


In [21]:
"""
for qid, q_dict in queries2.items():
    features_ordered = GAS(q_dict['feature_scores'], q_dict['similarity_dict'])
    q_dict['feature_order'] = features_ordered
"""

"\nfor qid, q_dict in queries2.items():\n    features_ordered = GAS(q_dict['feature_scores'], q_dict['similarity_dict'])\n    q_dict['feature_order'] = features_ordered\n"

In [125]:
def prepare_output(queries):
    y = []
    for q_dict in queries.values():
        y += q_dict['scores']
    return y

def prepare_input(queries, features_ordered, num_of_features=4):
    X = []
    for q_dict in queries.values():
        features_list_list = q_dict['features']
        for features_list in features_list_list:
            x_group = []
            for feature_index in features_ordered[:num_of_features]:
                feature_score = features_list[feature_index]
                x_group.append(feature_score)
            X.append(x_group)
    return X
    

In [126]:
y = prepare_output(train_queries)
y_test = prepare_output(test_queries)

y = np.array(y)
y_test = np.array(y_test)

In [127]:
def get_model(num_of_features):
    model = Sequential([
        Input(shape=(num_of_features,)),  # Define the input shape here
        Dense(128, activation='relu'),  # First hidden layer
        Dense(128, activation='relu'),  # Second hidden layer
        Dense(64, activation='relu'),   # Third hidden layer
        Dense(1)  # Output layer
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

In [128]:
best_score = -9999999999999999
best_model = None
best_num = None

for num_of_features in range(2,21):
    X = prepare_input(train_queries, features_ordered, num_of_features)
    X_test = prepare_input(test_queries, features_ordered, num_of_features)
    X = np.array(X)
    X_test = np.array(X_test)

    model = get_model(num_of_features)
    model.fit(X, y, epochs=100, verbose=0)

    y_pred = model.predict(X_test, verbose=0)
    ndcg_average = 0
    for index, q_dict in enumerate(test_queries.values()):
        pred_ys = y_pred[index*100: index * 100 + 100]
        documents = q_dict['bm25'].iloc
        real_ys = q_dict['scores']    
        # score doc_no doc_index
        scored_docs = [(pred_score, real_score, doc.docno, doc.docid)for pred_score, doc, real_score in zip(pred_ys, documents, real_ys)]
        sorted_docs = sorted(scored_docs, key=lambda x: x[0], reverse=True)
        sorted_relevances = [data[1] for data in sorted_docs]
        ndcg_score = ndcg_at_k(sorted_relevances)
        ndcg_average += ndcg_score
    ndcg_average /= len(test_queries)
    score = ndcg_average
    print(num_of_features ,ndcg_average)

    if score > best_score:
        best_score = score
        best_model = model
        best_num = num_of_features

2 0.4460593622064746
3 0.39460848063718146
4 0.4682815714569242
5 0.4545042318933759
6 0.44989135116749457
7 0.43757276904376313
8 0.4823585857797753
9 0.36754730066854363
10 0.3873129426716455
11 0.3376734208795327
12 0.2838450617238798
13 0.4044835325932972
14 0.45207375932870747
15 0.2371354963542922
16 0.33004508267195903
17 0.3777032131806791
18 0.33118383137780216
19 0.30902391133145907
20 0.38910604795958414


In [132]:
best_model.save('./gas_model.keras')

In [131]:
print(best_num)
print(best_score)
print(features_ordered)
"""
"""

8
0.4823585857797753
[19, 6, 5, 4, 18, 17, 2, 16, 9, 15, 0, 12, 14, 8, 1, 3, 10, 13, 7, 11]


'\n'

In [91]:
from tensorflow.keras.models import load_model
model = load_model('./gas_model.keras')
X = prepare_input(train_queries, features_ordered, 19)
X = np.array(X)
model.predict(X)

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  


array([[7.204371],
       [7.898884],
       [8.618757],
       ...,
       [8.654239],
       [8.960614],
       [9.192194]], dtype=float32)

[19, 6, 5, 4, 18, 17, 2, 16, 9, 15, 0, 12, 14, 8, 1, 3, 10, 13, 7, 11]

array([21.2268115, 11.       ,  2.       ,  0.       ])

In [ ]:
DQ_INPUT = 1
Q_INPUT = 2
D_INPUT = 3
feature_func_list = [
    (title_query_term_overlap, DQ_INPUT),
    (title_query_jaccard_similarity, DQ_INPUT),
    (title_query_dice_similarity, DQ_INPUT),
    (title_query_position, DQ_INPUT),
    (exact_match_title_query, DQ_INPUT),
    (query_length, Q_INPUT),
    (query_character_length, Q_INPUT),
    (term_overlap, DQ_INPUT),
    (jaccard_similarity, DQ_INPUT),
    (dice_similarity, DQ_INPUT),
    (term_overlap_extra, DQ_INPUT),
    (jaccard_similarity_extra, DQ_INPUT),
    (dice_similarity_extra, DQ_INPUT),
    (document_length, D_INPUT),
    (document_character_length, D_INPUT),
    (average_sentence_length, D_INPUT),
    (unique_word_count, D_INPUT),
    (url_depth, D_INPUT),
    (has_query_parameters, D_INPUT),
]
def get_ordered_features(doc, query):
    return [
        func(doc, query) if DQ_INPUT == input_type 
        else func(doc) if D_INPUT == input_type 
        else func(query) 
        for func, input_type in feature_func_list 
    ]

[1, 2, 3, 5]